# Track C — 04. miRNA Cleaning

The raw miRNA table is wide: one row per miRNA, 954 cell-line columns (expression values),
column names formatted as `celllinename_tissue`. This pipeline maps each column to a
`model_id` via `cell_line_lookup` (prefix-matching against `cell_line_name` /
`stripped_cell_line_name`, deduped by `model_id` to avoid false ambiguity), applies a
manual override for one confirmed synonym, drops columns with no resolvable match or
genuine ambiguity, then melts to long format `(mirna_id, mirna_symbol, model_id, expression_value)`.
Writes to `Track - C/outputs/mirna/`.

In [1]:
import os
import re
import pandas as pd
from data_utils import PARQUET_CLEAN, REF

PROJECT_ROOT = os.path.abspath(os.path.join(REF, '..'))
OUT_DIR      = os.path.join(PROJECT_ROOT, 'Track - C', 'outputs', 'mirna')
os.makedirs(OUT_DIR, exist_ok=True)


def normalize(s):
    if pd.isna(s):
        return None
    return re.sub(r'[^a-z0-9]', '', str(s).lower())

print('output dir:', OUT_DIR)


output dir: C:\Disertation\UoB-GeneTraceAI-25-26\Track - C\outputs\mirna


## 1. Column -> model_id resolution (deduped, iterative tissue-suffix stripping)

In [2]:
cell_line_lookup = pd.read_parquet(os.path.join(REF, 'cell_line_lookup.parquet'))
df = pd.read_parquet(os.path.join(PARQUET_CLEAN, 'mirna_clean.parquet'))
value_cols = [c for c in df.columns if c not in ['name', 'description']]

lookup_norm = {}
for _, row in cell_line_lookup.iterrows():
    for field in ['cell_line_name', 'stripped_cell_line_name']:
        if field in cell_line_lookup.columns and pd.notna(row[field]):
            key = normalize(row[field])
            lookup_norm.setdefault(key, set()).add(row['model_id'])

def resolve_column(col, lookup_norm, max_strip=5):
    parts = col.split('_')
    for strip_n in range(0, min(max_strip, len(parts))):
        candidate_name = '_'.join(parts[:len(parts) - strip_n]) if strip_n > 0 else col
        norm_name = normalize(candidate_name)
        if norm_name in lookup_norm:
            matches = lookup_norm[norm_name]
            if len(matches) == 1:
                return next(iter(matches)), strip_n, 'unique'
            return None, strip_n, f'ambiguous_{len(matches)}_matches'
    return None, None, 'no_match'

results = []
for col in value_cols:
    model_id, strip_n, status = resolve_column(col, lookup_norm)
    results.append({'column': col, 'model_id': model_id, 'strip_n': strip_n, 'status': status})

results_df = pd.DataFrame(results)
print(results_df['status'].value_counts())


status
unique                 947
ambiguous_2_matches      4
no_match                 3
Name: count, dtype: int64


## 2. Manual override + drop genuine holdouts

`d341med_central_nervous_system` -> `ACH-000095` (confirmed via lookup: `D341` shares the same
`stripped_cell_line_name` root). `ncih1339_lung` and `colo699_lung` are absent from the lookup
entirely (checked `cell_line_name`, `stripped_cell_line_name`, and `synonyms` — no hits) — not
worth more matching logic for 2 of 954 columns. The remaining ambiguous columns
(`tt_oesophagus`, `tt_thyroid`, `u251mg_central_nervous_system`,
`kmh2_haematopoietic_and_lymphoid_tissue`) are dropped rather than guessed.

In [3]:
manual_map = {
    'd341med_central_nervous_system': 'ACH-000095',
}
for col, mid in manual_map.items():
    results_df.loc[results_df['column'] == col, ['model_id', 'status']] = [mid, 'manual_override']

unresolved = results_df[results_df['model_id'].isna()]
print(f'Dropping {len(unresolved)} columns: {unresolved["column"].tolist()}')
unresolved.to_csv(os.path.join(OUT_DIR, 'mirna_unresolved_columns.csv'), index=False)

col_to_model = dict(zip(results_df['column'], results_df['model_id']))
resolved_cols = [c for c in value_cols if col_to_model.get(c) is not None]
print(f'Proceeding with {len(resolved_cols)} of {len(value_cols)} cell line columns')


Dropping 6 columns: ['ncih1339_lung', 'tt_oesophagus', 'u251mg_central_nervous_system', 'tt_thyroid', 'kmh2_haematopoietic_and_lymphoid_tissue', 'colo699_lung']
Proceeding with 948 of 954 cell line columns


## 3. Melt to long format + write output

In [4]:
id_cols = ['name', 'description']
melted = df[id_cols + resolved_cols].melt(
    id_vars=id_cols, value_vars=resolved_cols,
    var_name='raw_column', value_name='expression_value'
)
melted['model_id'] = melted['raw_column'].map(col_to_model)
melted = melted.rename(columns={'name': 'mirna_id', 'description': 'mirna_symbol'})
melted = melted.drop(columns=['raw_column'])

dupes = melted.duplicated(subset=['mirna_id', 'model_id']).sum()
print(f'Duplicate (mirna_id, model_id) pairs: {dupes}')
print(f'Final shape: {melted.shape}')
print(f'Unique miRNAs: {melted["mirna_id"].nunique()}')
print(f'Unique models: {melted["model_id"].nunique()}')

OUT_PATH = os.path.join(OUT_DIR, 'mirna_model_level.parquet')
melted.to_parquet(OUT_PATH)
print(f'\nSaved to {OUT_PATH}')


Duplicate (mirna_id, model_id) pairs: 0
Final shape: (695832, 4)
Unique miRNAs: 734
Unique models: 948

Saved to C:\Disertation\UoB-GeneTraceAI-25-26\Track - C\outputs\mirna\mirna_model_level.parquet
